> 在线讲义：[第 44 章 · ViT：把图片切成小块，当成一句话](https://wikieden.github.io/beyond-transformer/lessons/44-vit.html) · 本书《超越 Transformer》

# 第 44 章 · ViT：把图片切成小块，当成一句话

配套讲义：[在线版](https://wikieden.github.io/beyond-transformer/lessons/44-vit.html) · 仓库里 `beyond-transformer/notes/lessons/44-vit.md`（附录最后一节"代码对照"）

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的 Python 写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。只用 NumPy，不需要 GPU。练习题不在这里算，先在讲义里自己做。

In [ ]:
import numpy as np

np.set_printoptions(precision=4, suppress=True)  # 打印保留 4 位小数，不用科学计数法
print("numpy", np.__version__)

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。

| 讲义 | 公式 | 写法 | 跑出来 |
|---|---|---|---|
| 第一步 切小块 | $N = HW/P^2$ | `patchify(img, 2)` | 4 块：[1, 2, 3, 4]、[0, 1, 1, 0]、[0, 0, 1, 1]、[5, 6, 7, 8] |
| 第一步 序列多长 | $(224/P)^2$ | `seq_stats(224, 16)` | N = 196，每块 768 个数，序列 197，分数表 38809 |
| 第二步 投影 | $x_p E$ | `patches @ E` | [2.5, −4]、[0.5, 0]、[0.5, −2]、[6.5, −4] |
| 第二步 加 [class] 和位置 | 式 1 | `embed(patches, E, cls, pos)` | 5 × 2：第 1 行 [2.6, −3.9]，最后一行 [6.9, −3.6] |
| 第五步 硬蒸馏 | DeiT 式 3 | `hard_distill_loss(p, y, y_t)` | (0.693 + 0.916) / 2 ≈ 0.805 |

## 1. 先把要用的东西定义好

这几格只是定义，运行后没有输出也正常。

In [ ]:
IMG = np.array([[1, 2, 0, 1], [3, 4, 1, 0], [0, 0, 5, 6], [1, 1, 7, 8]], dtype=float)  # 讲义里编的 4 × 4 灰度图
E = np.array([[0.25, 1], [0.25, 1], [0.25, -1], [0.25, -1]])  # 第 1 列算平均亮度，第 2 列算上半减下半
CLS = np.zeros(2)  # [class] 向量，一开始设成 0
POS = np.array([[0, 0], [0.1, 0.1], [0.2, 0.2], [0.3, 0.3], [0.4, 0.4]])  # 编的位置编码，N + 1 = 5 行

**函数 `patchify`**：把图片按 P × P 切块，每块按行展平，从左到右、从上到下排好。

In [ ]:
def patchify(img, P):
    """把图片按 P × P 切块，每块按行展平，从左到右、从上到下排好。"""
    H, W = img.shape[:2]
    return np.array([img[i : i + P, j : j + P].reshape(-1) for i in range(0, H, P) for j in range(0, W, P)])

**函数 `seq_stats`**：size × size 的图切成 P × P 小块：块数 N、每块几个数、加 [class] 后序列长度、注意力分数表格子数。

In [ ]:
def seq_stats(size, P, C=3):
    """size × size 的图切成 P × P 小块：块数 N、每块几个数、加 [class] 后序列长度、注意力分数表格子数。"""
    N = (size // P) ** 2
    return {"N": N, "per_patch": P * P * C, "seq": N + 1, "scores": (N + 1) ** 2}

**函数 `embed`**：式 1：每块乘 E，最前面放 [class]，再加位置编码。

In [ ]:
def embed(patches, E, cls, pos):
    """式 1：每块乘 E，最前面放 [class]，再加位置编码。"""
    return np.vstack([cls, patches @ E]) + pos

**函数 `hard_distill_loss`**：DeiT 式 3：真实标签和老师标签的交叉熵各占一半。

In [ ]:
def hard_distill_loss(p, y, y_t):
    """DeiT 式 3：真实标签和老师标签的交叉熵各占一半。"""
    return 0.5 * -np.log(p[y]) + 0.5 * -np.log(p[y_t])

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("\n第一步 切小块：4 × 4 的图，P = 2")
patches = patchify(IMG, 2)
print(patches)
for P in (32, 16, 14, 8):
    print(f"224 × 224 彩色图，P = {P}：", seq_stats(224, P))

In [ ]:
print("\n第二步 线性投影：每块乘 E")
print(patches @ E)
print("加 [class] 和位置编码后的 z0（5 × 2）：")
print(embed(patches, E, CLS, POS))

In [ ]:
print("\n第五步 硬蒸馏：学生概率 [0.5, 0.1, 0.4]，真实标签 猫(0)，老师说 狐(2)")
p = np.array([0.5, 0.1, 0.4])
print(f"loss = {hard_distill_loss(p, 0, 2):.3f}")

## 3. 代码里才看得到的

- 彩色图的 `reshape(-1)` 顺序是"行 → 列 → 通道"，PyTorch 的 `Conv2d` 是"通道在前"，权重摆放顺序不同，但算的是同一件事。
- 下标从 0 数：[class] 是第 0 行，所以原文写 $z_L^0$。

## 4. 自己改一改

- 把 `seq_stats` 的 P 改成 8、32，核对第一步那张表。
- 把 E 的第 2 列改成 [1, −1, 1, −1]，它算的就变成"左列减右列"，看 4 块的第 2 个数怎么变。

在下面的空格子里写，或者直接改上面的格子。

In [ ]:
# 在这里写你的代码

## 口诀

> 图片切成邮票块，一块就是一个词；N 等于 HW 除 P 方，块小句长算得慢。
> 展平乘 E 变向量，位置编码也是学。
> 队首安排记录员，全看不遮取它输出。
> 没有看图的常识，数据不够就输给卷积；DeiT 加强增强请老师，ImageNet 也能训。